# PyTorch GPU and Device Handling

**PyTorch Interview Preparation · Notebook 08 · Estimated study time: 2–3 hours**

This notebook covers device selection, transfers, GPU memory, inference, mixed precision, timing, and practical debugging. It runs from top to bottom on CPU-only machines and uses CUDA or Apple MPS when available.

> **Core flow**
>
> detect hardware → choose device → move model → load CPU batch → move batch → compute → manage memory → move results only when needed → debug performance/OOM


## 1. Setup

Only PyTorch and the Python standard library are used. CUDA-only examples are guarded.


In [ ]:
import time

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

mps_available = (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
)
print("MPS available:", mps_available)


## 2. CPU vs GPU 🟢 Easy

| CPU | GPU |
|---|---|
| general-purpose processor | many parallel compute units |
| fewer powerful cores | excels at large tensor operations |
| strong for control flow/general computation | strong for matrix multiplication/neural networks |

```text
Large tensor operations + massive parallelism = GPU-friendly workload
```

A GPU is not automatically faster: tiny workloads, transfer overhead, and input bottlenecks can dominate.


## 3. CUDA and MPS 🟢 Easy

CUDA is NVIDIA's GPU-computing platform. PyTorch can use a CUDA-enabled NVIDIA GPU only when the build, compatible driver, and hardware are available.

```text
GPU ≠ automatically CUDA
CUDA → NVIDIA-specific
MPS → Apple Silicon GPU backend
```

### 🔥 Interview Essential: CUDA availability

Use `torch.cuda.is_available()` before CUDA-specific operations. Apple MPS availability is checked separately.


## 4. Selecting a Device 🟢 Easy

### 🔥 Interview Essential

Prefer CUDA, then MPS, then CPU. `torch.device` represents where tensors, parameters, and buffers should live.


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Selected device:", device)
print("Device type:", device.type)
assert device.type in {"cpu", "cuda", "mps"}

# Common shorter pattern (does not select Apple MPS):
cuda_or_cpu = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA-or-CPU pattern:", cuda_or_cpu)


## 5. Inspecting Tensor Devices 🟢 Easy

Tensors are created on CPU unless a device is specified. Inspect placement with `.device`.


In [ ]:
x_cpu = torch.randn(3, 4)
x_direct = torch.randn(3, 4, device=device)
print("Default tensor:", x_cpu.device)
print("Direct creation:", x_direct.device)
assert x_cpu.device.type == "cpu"
assert x_direct.device.type == device.type


## 6. Moving Tensors Between Devices 🟢 Easy

### 🔥 Interview Essential: `.to(device)`

`.to(...)` returns a tensor with the requested placement/dtype. Do not assume it mutates the original reference.

```python
x = x.to(device)       # portable: CPU, CUDA, MPS
x = x.cpu()            # move to CPU
x = x.cuda()           # NVIDIA-specific; fails without CUDA
```

Create on CPU and move later, or create directly on the target device. `.to(device)` is usually preferred over hardcoded `.cuda()` in portable code.


In [ ]:
# Exercise 1 — move a tensor and validate the result.
x = torch.randn(5, 10)
answer = x.to(device)

assert isinstance(answer, torch.Tensor)
assert answer.device.type == device.type
print("✅ Exercise 1 correct")

# Exercise 2 — prove .to is not an in-place placement operation.
original = torch.randn(2, 2)
moved = original.to(device)
assert original.device.type == "cpu"
assert moved.device.type == device.type
print("original:", original.device, "returned tensor:", moved.device)


## 7. Moving Models to a Device 🟢 Easy

### 🔥 Interview Essential

`model.to(device)` moves parameters and registered buffers. Move a model once before the loop—not once per batch.


In [ ]:
model = nn.Linear(10, 3)
model = model.to(device)
print("Model parameter devices:")
for parameter in model.parameters():
    print(parameter.device)
assert all(p.device.type == device.type for p in model.parameters())

def get_model_device(model):
    """Useful for ordinary models; parameterless modules need another strategy."""
    return next(model.parameters()).device

assert get_model_device(model).type == device.type


## 8. Device Mismatch Errors 🟢 Easy

### 🔥 Very Common Interview / Debugging Issue

```text
MODEL DEVICE = INPUT DEVICE = TARGET DEVICE
```

If a CUDA model receives a CPU input, an operation usually fails with text like `Expected all tensors to be on the same device`. PyTorch does not silently move the input. The target commonly must match too because the loss combines logits and target.

**Quick inspection pattern:** print `next(model.parameters()).device`, `X.device`, and `y.device`. The example below demonstrates the diagnosis without deliberately crashing.


In [ ]:
debug_model = nn.Linear(4, 2).to(device)
debug_X_cpu = torch.randn(8, 4)
debug_y_cpu = torch.randint(0, 2, (8,))

print("Model device:", next(debug_model.parameters()).device)
print("Input device:", debug_X_cpu.device)
print("Target device:", debug_y_cpu.device)

debug_X = debug_X_cpu.to(device)
debug_y = debug_y_cpu.to(device)
debug_logits = debug_model(debug_X)
debug_loss = nn.CrossEntropyLoss()(debug_logits, debug_y)
assert debug_logits.device.type == device.type
print("✅ Corrected forward/loss:", debug_loss.item())


## 9. Device-Agnostic Code 🟢 Easy

### 🔥 Interview Essential: correct training pattern

Pass `device` explicitly, move model once, and move each CPU batch before computation.


In [ ]:
def train_step(model, X, y, loss_fn, optimizer, device):
    model.train()
    X = X.to(device)
    y = y.to(device)
    optimizer.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()
    return loss.item()

step_model = nn.Linear(6, 3).to(device)
step_optimizer = optim.SGD(step_model.parameters(), lr=0.1)
step_loss = train_step(
    step_model, torch.randn(16, 6), torch.randint(0, 3, (16,)),
    nn.CrossEntropyLoss(), step_optimizer, device,
)
assert isinstance(step_loss, float)
print("One device-agnostic loss:", step_loss)


## 10. Moving Data Back to CPU 🟢 Easy

Use `.cpu()` before CPU-only libraries, some logging/serialization, or NumPy. For a tensor that may track gradients, the safe conversion is:

```python
array = tensor.detach().cpu().numpy()
```

`detach()` removes Autograd history, `cpu()` ensures CPU memory, and `numpy()` converts to an ndarray. **🔥 Interview Useful**


In [ ]:
prediction = debug_model(debug_X)
prediction_array = prediction.detach().cpu().numpy()
assert prediction_array.shape == (8, 2)
print("NumPy shape:", prediction_array.shape)


## 11. CPU–GPU Transfer Cost 🟡 Medium

### 🔥 Interview Essential

Moving data is not free:

```text
CPU memory → transfer → GPU memory → computation → transfer → CPU memory
```

Move a batch once, perform as much work as possible, and return only required results. Repeated `.cpu()`/`.to(device)` calls can erase GPU gains.

**Exercise 3:** Diagnose `X→device; output=model(X); output→CPU; output→device`. **Answer:** the last two moves are an unnecessary round trip.


## 12. DataLoader and GPU Transfer 🟢 Easy

A normal DataLoader yields CPU tensors:

```text
Dataset → DataLoader → CPU batch → .to(device) → GPU/MPS model
```


In [ ]:
loader_dataset = TensorDataset(torch.randn(96, 6), torch.randint(0, 3, (96,)))
train_loader = DataLoader(loader_dataset, batch_size=16, shuffle=True)
batch_X, batch_y = next(iter(train_loader))
assert batch_X.device.type == "cpu" and batch_y.device.type == "cpu"
batch_X = batch_X.to(device)
batch_y = batch_y.to(device)
assert batch_X.device.type == device.type and batch_y.device.type == device.type
print("✅ DataLoader CPU batch moved to", device)


## 13. `pin_memory` 🟡 Medium

### 🔥 Interview Essential

Pinned (page-locked) CPU memory can improve CPU→CUDA transfers. It does **not** move data to GPU and is primarily useful for CUDA. A portable choice is `pin_memory=torch.cuda.is_available()`.


In [ ]:
pinned_loader = DataLoader(
    loader_dataset,
    batch_size=64,
    shuffle=True,
    pin_memory=torch.cuda.is_available(),
)
assert pinned_loader.pin_memory == torch.cuda.is_available()
print("pin_memory:", pinned_loader.pin_memory)


## 14. `non_blocking` Transfers 🟡 Medium

### 🔥 Interview Essential

With suitable pinned-memory CUDA transfers, `non_blocking=True` can permit overlap with other work. It is a permission/hint, not a guarantee of speedup; benchmark the full pipeline.


In [ ]:
for transfer_X, transfer_y in pinned_loader:
    transfer_X = transfer_X.to(
        device, non_blocking=torch.cuda.is_available()
    )
    transfer_y = transfer_y.to(
        device, non_blocking=torch.cuda.is_available()
    )
    assert transfer_X.device.type == device.type
    break
print("✅ Portable transfer pattern works")


## 15. GPU Memory Basics 🟢 Easy

### 🔥 Interview Essential

```text
Training memory = parameters + activations + gradients + optimizer state
                + input batch + temporary buffers
```

Model weights are only one component. Deep networks, large images/sequences, and large batches can make activations dominant. Adam/AdamW also keep per-parameter optimizer states; no universal exact multiplier applies across precision and implementations.


## 16. CUDA Memory Inspection 🟡 Medium

- `memory_allocated()` → bytes currently occupied by live CUDA tensors
- `memory_reserved()` → bytes held by PyTorch's caching allocator
- `max_memory_allocated()` → peak live-tensor allocation


In [ ]:
def bytes_to_mb(value):
    return value / 1024**2

if torch.cuda.is_available():
    print("Allocated MB:", bytes_to_mb(torch.cuda.memory_allocated()))
    print("Reserved MB:", bytes_to_mb(torch.cuda.memory_reserved()))
    print("Peak allocated MB:", bytes_to_mb(torch.cuda.max_memory_allocated()))
else:
    print("CUDA memory metrics unavailable on this device.")


### `empty_cache()` and Python references 🔥 Common Misconception

`torch.cuda.empty_cache()` releases **unused cached** CUDA memory so it may be available to other applications. It does not delete live tensors or increase the memory available to PyTorch for tensors that are still referenced.

```python
del large_tensor
if torch.cuda.is_available():
    torch.cuda.empty_cache()
```

Deleting a reference makes storage eligible for reuse. Manual cleanup is not required for every tensor.


In [ ]:
temporary = torch.ones(8, device=device)
del temporary
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ Safe conditional cache call")


## 17. Out-of-Memory Errors 🟢 Easy

### 🔥 Interview Essential: CUDA OOM

Common causes: batch/model/activations too large, retained computation graphs, storing GPU tensors each iteration, multiple models, large sequence/image sizes, or unnecessary tensors.

**First-response checklist:**

1. Reduce batch size.
2. Use mixed precision where supported.
3. Use `no_grad()`/`inference_mode()` for inference.
4. Avoid retaining unnecessary GPU tensors/graphs.
5. Delete unused large objects; clear unused cache if useful to other processes.
6. Use gradient accumulation for a large effective batch.
7. Consider gradient checkpointing.
8. Reduce sequence/image size and inspect leaks.

For transformers, longer sequences consume more activation memory; attention can become especially expensive.


### Graph-retention bugs 🔥 Important Debugging Pattern

Bad: `loss_history.append(loss)` can retain the graph. Prefer `loss.item()` for scalar history, or `loss.detach().cpu()` when a tensor is required. Likewise, saving `output` during training can retain large GPU tensors/graphs; use `output.detach().cpu()` when gradients are unnecessary.


In [ ]:
# Exercise 4 — safe loss history.
history_model = nn.Linear(6, 3).to(device)
history_loss = nn.CrossEntropyLoss()(
    history_model(torch.randn(4, 6, device=device)),
    torch.randint(0, 3, (4,), device=device),
)
loss_history = []
loss_history.append(history_loss.item())
assert isinstance(loss_history[0], float)
print("✅ History does not retain the loss graph")


## 18. Batch Size and Memory 🟢 Easy

### 🔥 Interview Essential

```text
larger batch → more simultaneous samples → more activations → more memory
```

If batch 128 OOMs, the first easy experiment is a smaller batch. Gradient accumulation processes small physical batches while approximating a larger effective batch. Do not maximize batch size blindly: measure end-to-end throughput.

**OOM exercise:** batch 32 works and 256 fails. Why? More activation/input memory. First step? Reduce it. Accumulation? Divide the logical batch across several forward/backward passes before stepping.


## 19. Training vs Inference Memory 🟢 Easy

### 🔥 Interview Essential

| Training | Inference |
|---|---|
| parameters | parameters |
| forward activations saved for backward | forward intermediates only as needed |
| gradients | no gradients |
| optimizer state | no optimizer state |

Inference therefore usually needs less memory.


## 20. `torch.no_grad()` and Inference 🟢 Easy

### 🔥 Interview Essential

`no_grad()` disables gradient calculation, reducing graph storage and overhead. `inference_mode()` goes further with inference-specific Autograd optimizations, but the modes are not interchangeable in every edge case. Always use `model.eval()` too: it changes module behavior (for example dropout/batch norm), whereas the contexts change Autograd tracking.


In [ ]:
inference_model = nn.Sequential(nn.Linear(6, 12), nn.ReLU(), nn.Linear(12, 3)).to(device)
inference_X = torch.randn(5, 6, device=device)
inference_model.eval()

with torch.no_grad():
    no_grad_output = inference_model(inference_X)
with torch.inference_mode():
    inference_output = inference_model(inference_X)

assert not no_grad_output.requires_grad
assert not inference_output.requires_grad
print("✅ Inference outputs do not track gradients")


## 21. Floating-Point Precision 🟢 Easy

| dtype | Bits | Storage vs FP32 | General note |
|---|---:|---:|---|
| `float32` | 32 | 1× | common default |
| `float16` | 16 | ~0.5× | narrower numeric range; can under/overflow |
| `bfloat16` | 16 | ~0.5× | FP32-like exponent range; less mantissa precision |

Hardware and operation support determine actual speedups.


In [ ]:
precision_x = torch.randn(10)
precision_half = precision_x.to(torch.float16)
precision_bf16 = precision_x.to(torch.bfloat16)
assert precision_x.dtype == torch.float32
assert precision_half.element_size() == 2
assert precision_bf16.element_size() == 2
print(precision_x.dtype, precision_half.dtype, precision_bf16.dtype)


## 22. `float16` 🟡 Medium

FP16 uses less storage and can be faster on appropriate GPUs, but its smaller exponent range creates underflow/overflow risk. Blindly converting every operation can be numerically unstable.


## 23. `bfloat16` 🟡 Medium

### 🔥 Interview Essential: FP16 vs BF16

```text
float16  → more mantissa precision, much smaller exponent range
bfloat16 → less mantissa precision, much larger exponent range
```

BF16 is common in modern transformer training because its range is often more robust, when hardware supports it. Speed and support remain device-specific.


## 24. Mixed Precision 🟡 Medium

### 🔥 Interview Essential

Mixed precision uses lower precision for operations where it is safe and higher precision where stability needs it. Benefits can include less memory, faster compute, and larger batches; numerical stability remains a concern.


## 25. `autocast` 🟡 Medium

Autocast chooses operation-specific dtypes inside its context. This notebook executes the CUDA FP16 form only when CUDA exists.


In [ ]:
amp_model = nn.Linear(6, 3).to(device)
amp_X = torch.randn(8, 6, device=device)
amp_y = torch.randint(0, 3, (8,), device=device)

if torch.cuda.is_available():
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        amp_logits = amp_model(amp_X)
        amp_loss = nn.CrossEntropyLoss()(amp_logits, amp_y)
    print("CUDA autocast loss:", amp_loss.item())
else:
    print("CUDA unavailable — showing mixed-precision syntax only.")


## 26. Gradient Scaling 🟡 Medium

FP16 gradients can underflow toward zero. `torch.amp.GradScaler("cuda")` scales the loss before backward, unscales before the optimizer update, adjusts the scale over time, and helps preserve small gradients. BF16's larger exponent range means scaling is often less necessary, though exact practice depends on the setup.


## 27. Mixed Precision Training Pattern 🟡 Medium

The current PyTorch API is `torch.amp.GradScaler("cuda")`:

```python
scaler = torch.amp.GradScaler("cuda")
optimizer.zero_grad()
with torch.autocast(device_type="cuda", dtype=torch.float16):
    logits = model(X)
    loss = loss_fn(logits, y)
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
```


In [ ]:
if torch.cuda.is_available():
    scaler = torch.amp.GradScaler("cuda")
    amp_optimizer = optim.SGD(amp_model.parameters(), lr=0.01)
    amp_optimizer.zero_grad()
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        amp_logits = amp_model(amp_X)
        amp_loss = nn.CrossEntropyLoss()(amp_logits, amp_y)
    scaler.scale(amp_loss).backward()
    scaler.step(amp_optimizer)
    scaler.update()
    print("✅ Modern CUDA AMP step completed")
else:
    print("CUDA AMP step skipped safely.")


### Mixed-precision inference

```python
model.eval()
with torch.inference_mode():
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        output = model(X)
```

On suitable hardware this can reduce memory and improve inference speed.


## 28. Performance Timing 🟡 Medium

For CPU work, `time.perf_counter()` is sufficient. Warm up first and measure repeated iterations to reduce one-time setup noise.


In [ ]:
timing_x = torch.randn(256, 256)
for _ in range(2):
    _ = timing_x @ timing_x

start = time.perf_counter()
for _ in range(5):
    timing_y = timing_x @ timing_x
elapsed = time.perf_counter() - start
print(f"CPU elapsed for 5 matmuls: {elapsed:.6f} s")
assert timing_y.shape == (256, 256)


## 29. Synchronization 🟡 Medium

CUDA launches are asynchronous. A Python timer may stop before GPU work finishes, so synchronize immediately before and after the measured region. Avoid unnecessary synchronization in normal training because it prevents overlap. CUDA events are often a better GPU timer.


In [ ]:
if torch.cuda.is_available():
    cuda_model = nn.Linear(128, 128).cuda()
    cuda_x = torch.randn(64, 128, device="cuda")
    for _ in range(3):
        _ = cuda_model(cuda_x)  # warm-up
    torch.cuda.synchronize()
    start = time.perf_counter()
    _ = cuda_model(cuda_x)
    torch.cuda.synchronize()
    print("Synchronized CUDA seconds:", time.perf_counter() - start)

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    start_event.record(); _ = cuda_model(cuda_x); end_event.record()
    torch.cuda.synchronize()
    print("CUDA-event milliseconds:", start_event.elapsed_time(end_event))
else:
    print("CUDA timing skipped safely.")


## 30. Common Performance Mistakes 🟡 Medium

- excessive CPU↔GPU transfers
- slow DataLoader/preprocessing/storage
- batch too small or model too tiny to amortize overhead
- frequent `torch.cuda.synchronize()`, `.item()`, or printing CUDA tensors
- timing without warm-up/synchronization
- assuming FP16 is always faster

`loss.item()` on CUDA may synchronize. It is fine for necessary logging, but per-batch synchronization-heavy logging can reduce throughput. A GPU can be slower than a CPU for a tiny workload because launches/transfers dominate.

```text
CPU/storage → DataLoader → CPU batch → transfer → GPU → model compute
```

A bottleneck anywhere reduces utilization.


## 31. Common Device Mistakes

1. Model on CUDA, input on CPU.
2. Logits on CUDA, target on CPU.
3. Calling `X.to(device)` without assigning the returned tensor.
4. Moving the model inside every batch.
5. Hardcoding CUDA without availability checks.
6. Using `.cuda()` in MPS/CPU-portable code.
7. Calling `.numpy()` on a CUDA tensor.
8. Calling `.numpy()` on a tensor requiring gradients.
9. Repeated CPU/GPU transfers.
10. Assuming pinned memory moves data to GPU.
11. Assuming `empty_cache()` frees live tensors.
12. Saving graph-connected GPU tensors each iteration.
13. Tracking gradients during inference.
14. Increasing batch size until OOM without measuring throughput.
15. Assuming FP16 is always faster.
16. Using low precision without hardware checks.
17. Benchmarking CUDA without synchronization.


## 32. Debugging Exercises

For each scenario, state the likely issue and fix before revealing the answer.


### Debug 1 · Model is on CUDA; `X` from the DataLoader goes directly into `model(X)`.

<details><summary>Diagnosis and fix</summary>

DataLoader batches are normally CPU tensors. Assign `X = X.to(device)` before the forward pass.

</details>


### Debug 2 · `X.to(device)` is called, then a CUDA model receives `X`.

<details><summary>Diagnosis and fix</summary>

`.to` is not an in-place placement operation. Use `X = X.to(device)`.

</details>


### Debug 3 · `pred.numpy()` fails after a CUDA forward pass.

<details><summary>Diagnosis and fix</summary>

Use `pred.detach().cpu().numpy()` when gradients may be tracked.

</details>


### Debug 4 · Device variable falls back to CPU, but code calls `model.cuda()`.

<details><summary>Diagnosis and fix</summary>

The model move ignores the chosen device and fails without CUDA. Use `model.to(device)`.

</details>


### Debug 5 · `losses.append(loss)` causes memory growth over many iterations.

<details><summary>Diagnosis and fix</summary>

The graph-connected loss is retained. Store `loss.item()` or a detached CPU tensor.

</details>


### Debug 6 · `all_outputs.append(output)` grows CUDA memory during training.

<details><summary>Diagnosis and fix</summary>

Saved outputs retain GPU storage and possibly graphs. If gradients are unnecessary, append `output.detach().cpu()`.

</details>


### Debug 7 · Logits are CUDA and the CrossEntropyLoss target is CPU.

<details><summary>Diagnosis and fix</summary>

Move the target: `y = y.to(device)`.

</details>


### Debug 8 · A model is moved to the device inside every batch.

<details><summary>Diagnosis and fix</summary>

Move it once before the loop to avoid repeated work and confusing optimizer state.

</details>


### Debug 9 · `pin_memory=True` is set, but batches remain on CPU.

<details><summary>Diagnosis and fix</summary>

Expected: pinning does not move tensors. Still call `.to(device)`.

</details>


### Debug 10 · `empty_cache()` does not reduce allocated memory while `x` exists.

<details><summary>Diagnosis and fix</summary>

`x` is live. Clear references first; cache clearing only releases unused cached blocks.

</details>


### Debug 11 · Inference unexpectedly stores graphs.

<details><summary>Diagnosis and fix</summary>

Call `model.eval()` and run under `torch.inference_mode()` or `no_grad()`.

</details>


### Debug 12 · CUDA timer reports implausibly tiny times.

<details><summary>Diagnosis and fix</summary>

GPU work is asynchronous. Warm up and synchronize, or use CUDA events.

</details>


### Debug 13 · GPU utilization is low and it waits between batches.

<details><summary>Diagnosis and fix</summary>

Inspect data loading, preprocessing, storage, worker count, batch size, and transfer overlap.

</details>


### Debug 14 · Batch 32 works; batch 256 OOMs.

<details><summary>Diagnosis and fix</summary>

The larger batch creates more activations. Reduce physical batch or accumulate gradients.

</details>


### Debug 15 · Training OOM appears only after hundreds of iterations.

<details><summary>Diagnosis and fix</summary>

Suspect retained tensors/graphs in histories, outputs, callbacks, or closures.

</details>


### Debug 16 · A tiny MLP is slower on GPU than CPU.

<details><summary>Diagnosis and fix</summary>

Kernel-launch and transfer overhead can dominate small workloads.

</details>


### Debug 17 · FP16 training produces NaNs.

<details><summary>Diagnosis and fix</summary>

Check numerical range, autocast boundaries, GradScaler use, learning rate, and hardware support.

</details>


### Debug 18 · `non_blocking=True` gives no speedup.

<details><summary>Diagnosis and fix</summary>

It is not a guarantee; confirm CUDA, pinned source memory, overlap opportunity, and end-to-end timing.

</details>


### Debug 19 · A parameter device check crashes for a parameterless module.

<details><summary>Diagnosis and fix</summary>

`next(model.parameters())` assumes a parameter exists; inspect inputs/buffers or handle `StopIteration`.

</details>


### Debug 20 · Frequent `print(loss)` slows CUDA training.

<details><summary>Diagnosis and fix</summary>

Formatting/reading CUDA values may synchronize; reduce per-step logging.

</details>


### Device state inspection drill 🟢 Easy

Memorize this first diagnostic:


In [ ]:
inspection_model = nn.Linear(6, 3).to(device)
inspection_X = torch.randn(2, 6).to(device)
inspection_y = torch.randint(0, 3, (2,)).to(device)
print("Model device:", next(inspection_model.parameters()).device)
print("Input device:", inspection_X.device)
print("Target device:", inspection_y.device)
assert len({next(inspection_model.parameters()).device.type,
            inspection_X.device.type, inspection_y.device.type}) == 1


### Tensor and parameter memory estimation 🔥 Interview Useful

Memory bytes = `numel() * element_size()`. A `(1000, 1000)` FP32 tensor has 1,000,000 elements × 4 bytes = 4,000,000 bytes ≈ 3.81 MiB.


In [ ]:
memory_x = torch.empty(1000, 1000, dtype=torch.float32)
memory_bytes = memory_x.numel() * memory_x.element_size()
print(memory_bytes, "bytes =", bytes_to_mb(memory_bytes), "MiB")
assert memory_bytes == 4_000_000

def parameter_memory_bytes(model):
    return sum(p.numel() * p.element_size() for p in model.parameters())

memory_model = nn.Sequential(nn.Linear(10, 20), nn.Linear(20, 3))
param_bytes = parameter_memory_bytes(memory_model)
expected_params = (10 * 20 + 20) + (20 * 3 + 3)
assert param_bytes == expected_params * 4
print("Parameter-only memory:", param_bytes, "bytes")


## 33. Interview Questions

Say a short answer aloud first, then open the explanation.


### 1. How do you check whether CUDA is available?

<details><summary>Short Interview Answer</summary>

Call `torch.cuda.is_available()`.

</details>

<details><summary>Detailed Explanation</summary>

It confirms that the current PyTorch runtime can use a compatible CUDA device; a physical GPU alone is insufficient.

</details>


### 2. How do you select CPU or GPU dynamically?

<details><summary>Short Interview Answer</summary>

Build a `torch.device` after availability checks.

</details>

<details><summary>Detailed Explanation</summary>

Common portable priority is CUDA, then MPS, then CPU.

</details>


### 3. What does `.to(device)` do?

<details><summary>Short Interview Answer</summary>

It returns a tensor or module placed on the requested device.

</details>

<details><summary>Detailed Explanation</summary>

For tensors assign the return value; for modules it moves parameters and registered buffers and also returns the module.

</details>


### 4. Why must model and input use the same device?

<details><summary>Short Interview Answer</summary>

Their tensors participate in the same operations.

</details>

<details><summary>Detailed Explanation</summary>

A kernel cannot generally combine CPU storage and CUDA/MPS storage, so PyTorch reports a mismatch.

</details>


### 5. How do you move a model to GPU?

<details><summary>Short Interview Answer</summary>

After checking CUDA, use `model = model.to(device)`.

</details>

<details><summary>Detailed Explanation</summary>

This is more portable than hardcoding `model.cuda()`.

</details>


### 6. How do you move a tensor back to CPU?

<details><summary>Short Interview Answer</summary>

Use `x = x.cpu()`.

</details>

<details><summary>Detailed Explanation</summary>

This is needed before NumPy conversion and many CPU-only consumers.

</details>


### 7. `.cuda()` versus `.to(device)`?

<details><summary>Short Interview Answer</summary>

`.cuda()` is CUDA-specific; `.to(device)` is portable.

</details>

<details><summary>Detailed Explanation</summary>

A selected device may represent CPU, CUDA, or MPS and can also include an index.

</details>


### 8. Why is `.to(device)` more portable?

<details><summary>Short Interview Answer</summary>

It avoids hardcoding an NVIDIA backend.

</details>

<details><summary>Detailed Explanation</summary>

The same code follows the runtime-selected CPU/CUDA/MPS device.

</details>


### 9. What is MPS in PyTorch?

<details><summary>Short Interview Answer</summary>

Apple's Metal-backed accelerator backend.

</details>

<details><summary>Detailed Explanation</summary>

It enables supported PyTorch operations on Apple Silicon GPUs and is selected with device type `mps`.

</details>


### 10. What causes 'Expected all tensors to be on the same device'?

<details><summary>Short Interview Answer</summary>

An operation received tensors on different devices.

</details>

<details><summary>Detailed Explanation</summary>

Print model, input, target, and any newly created tensor devices to find the mismatch.

</details>


### 11. Why are CPU-to-GPU transfers expensive?

<details><summary>Short Interview Answer</summary>

They move bytes across a device interconnect.

</details>

<details><summary>Detailed Explanation</summary>

Latency and bandwidth costs can dominate small computations or repeated round trips.

</details>


### 12. Why avoid unnecessary device transfers?

<details><summary>Short Interview Answer</summary>

They waste time and may synchronize work.

</details>

<details><summary>Detailed Explanation</summary>

Keep batches/results on the accelerator across related operations.

</details>


### 13. Where are DataLoader batches usually created?

<details><summary>Short Interview Answer</summary>

In CPU memory.

</details>

<details><summary>Detailed Explanation</summary>

The training loop normally transfers each batch to the selected accelerator.

</details>


### 14. What does `pin_memory=True` do?

<details><summary>Short Interview Answer</summary>

It asks the DataLoader to place batches in pinned CPU memory.

</details>

<details><summary>Detailed Explanation</summary>

Pinned memory can support faster/asynchronous host-to-CUDA copies.

</details>


### 15. Does pinned memory move data to GPU?

<details><summary>Short Interview Answer</summary>

No.

</details>

<details><summary>Detailed Explanation</summary>

The loop must still call `.to(device)`; pinning only changes host-memory properties.

</details>


### 16. What does `non_blocking=True` do?

<details><summary>Short Interview Answer</summary>

It permits an asynchronous copy when conditions support it.

</details>

<details><summary>Detailed Explanation</summary>

Pinned source memory and CUDA are typical ingredients; speedup is not guaranteed.

</details>


### 17. What consumes GPU memory during training?

<details><summary>Short Interview Answer</summary>

Weights, activations, gradients, optimizer state, batches, and temporaries.

</details>

<details><summary>Detailed Explanation</summary>

Activations or optimizer states can be as important as model parameters.

</details>


### 18. Why does training need more memory than inference?

<details><summary>Short Interview Answer</summary>

Training saves backward activations and keeps gradients/optimizer state.

</details>

<details><summary>Detailed Explanation</summary>

Inference generally omits these and can disable graph construction.

</details>


### 19. How does `torch.no_grad()` help inference?

<details><summary>Short Interview Answer</summary>

It disables gradient recording.

</details>

<details><summary>Detailed Explanation</summary>

That avoids storing many backward intermediates and reduces compute overhead.

</details>


### 20. What is `torch.inference_mode()`?

<details><summary>Short Interview Answer</summary>

A stronger inference-oriented no-grad mode.

</details>

<details><summary>Detailed Explanation</summary>

It removes additional Autograd overhead but has stricter semantics in some edge cases.

</details>


### 21. What commonly causes CUDA OOM?

<details><summary>Short Interview Answer</summary>

A working set larger than available GPU memory or retained tensors/graphs.

</details>

<details><summary>Detailed Explanation</summary>

Batch, activations, model, optimizer, sequence length, and leaks all matter.

</details>


### 22. What is the first thing to try after OOM?

<details><summary>Short Interview Answer</summary>

Reduce physical batch size.

</details>

<details><summary>Detailed Explanation</summary>

It directly reduces batch/activation memory and is quick to test.

</details>


### 23. How does batch size affect memory?

<details><summary>Short Interview Answer</summary>

Larger batches usually require more activation and input memory.

</details>

<details><summary>Detailed Explanation</summary>

The relationship can be close to linear for many activation components.

</details>


### 24. How does sequence length affect memory?

<details><summary>Short Interview Answer</summary>

Longer sequences increase activation memory.

</details>

<details><summary>Detailed Explanation</summary>

Transformer attention can grow especially quickly as sequence length rises.

</details>


### 25. What does `torch.cuda.empty_cache()` do?

<details><summary>Short Interview Answer</summary>

It releases unused cached CUDA blocks.

</details>

<details><summary>Detailed Explanation</summary>

It may help other applications or fragmentation situations, but does not delete tensors.

</details>


### 26. Does `empty_cache()` free referenced tensors?

<details><summary>Short Interview Answer</summary>

No.

</details>

<details><summary>Detailed Explanation</summary>

Live Python references keep the underlying allocation in use.

</details>


### 27. What is `memory_allocated()`?

<details><summary>Short Interview Answer</summary>

CUDA memory occupied by live tensors.

</details>

<details><summary>Detailed Explanation</summary>

It differs from allocator-reserved memory.

</details>


### 28. What is `memory_reserved()`?

<details><summary>Short Interview Answer</summary>

CUDA memory held by PyTorch's caching allocator.

</details>

<details><summary>Detailed Explanation</summary>

Some reserved blocks can be currently unused but ready for reuse.

</details>


### 29. What is mixed precision?

<details><summary>Short Interview Answer</summary>

Using lower and higher floating-point precision selectively.

</details>

<details><summary>Detailed Explanation</summary>

Autocast chooses suitable op precision; scaling supports FP16 gradients.

</details>


### 30. Why use mixed precision?

<details><summary>Short Interview Answer</summary>

To reduce memory and potentially improve throughput.

</details>

<details><summary>Detailed Explanation</summary>

Benefits depend on hardware and numerical behavior.

</details>


### 31. FP16 versus BF16?

<details><summary>Short Interview Answer</summary>

FP16 has more mantissa bits; BF16 has a much larger exponent range.

</details>

<details><summary>Detailed Explanation</summary>

BF16 is often more range-robust for deep learning; both use two-byte storage.

</details>


### 32. Why can FP16 be unstable?

<details><summary>Short Interview Answer</summary>

Its narrow numeric range can underflow or overflow.

</details>

<details><summary>Detailed Explanation</summary>

Autocast and gradient scaling mitigate common problems but do not fix every instability.

</details>


### 33. What does autocast do?

<details><summary>Short Interview Answer</summary>

It selects operation-specific dtypes inside a context.

</details>

<details><summary>Detailed Explanation</summary>

It keeps sensitive operations in safer precision rather than converting everything blindly.

</details>


### 34. Why scale gradients in FP16 training?

<details><summary>Short Interview Answer</summary>

To keep tiny gradients representable during backward.

</details>

<details><summary>Detailed Explanation</summary>

The scaler later unscales and dynamically updates the scale.

</details>


### 35. Is scaling as important for BF16?

<details><summary>Short Interview Answer</summary>

Often less so because BF16 has a larger exponent range.

</details>

<details><summary>Detailed Explanation</summary>

Exact practice depends on the framework, model, hardware, and training behavior.

</details>


### 36. Why can a Python CUDA timer mislead?

<details><summary>Short Interview Answer</summary>

CUDA work is queued asynchronously.

</details>

<details><summary>Detailed Explanation</summary>

Synchronize around the interval or use CUDA events after warm-up.

</details>


### 37. What does `torch.cuda.synchronize()` do?

<details><summary>Short Interview Answer</summary>

It waits for queued CUDA work to finish.

</details>

<details><summary>Detailed Explanation</summary>

It makes timing accurate but excessive use can reduce overlap/performance.

</details>


### 38. Why can frequent `.item()` calls slow training?

<details><summary>Short Interview Answer</summary>

Reading a CUDA scalar can force CPU-GPU synchronization.

</details>

<details><summary>Detailed Explanation</summary>

Log when useful, but avoid synchronization-heavy per-step behavior.

</details>


### 39. How do you diagnose low GPU utilization?

<details><summary>Short Interview Answer</summary>

Inspect input pipeline, workload size, transfers, and synchronization.

</details>

<details><summary>Detailed Explanation</summary>

DataLoader/preprocessing/storage bottlenecks, small batches/models, and Python overhead are common.

</details>


### 40. How do you estimate tensor memory?

<details><summary>Short Interview Answer</summary>

Multiply `numel()` by `element_size()`.

</details>

<details><summary>Detailed Explanation</summary>

This calculates tensor storage, not graph, allocator, or model-wide overhead.

</details>


## 34. Knowledge Check Quiz

Choose one answer for each question. Answers follow the quiz.


**Q1. Most portable tensor move?**

A. `x.cuda()`  
B. `x.gpu()`  
C. `x.to(device)`  
D. `torch.move(x)`


**Q2. CUDA model receives CPU X. Likely result?**

A. Automatic move  
B. CPU execution  
C. Device mismatch  
D. Gradients off


**Q3. Usually uses more GPU memory?**

A. Inference  
B. Training  
C. Same always  
D. DataLoader


**Q4. About half FP32 storage?**

A. int64  
B. float16  
C. float64  
D. complex64


**Q5. Apple Silicon accelerator backend?**

A. CUDA  
B. ROCm  
C. MPS  
D. XLA


**Q6. Normal DataLoader batch location?**

A. CPU  
B. CUDA  
C. MPS  
D. Chosen automatically


**Q7. Does pinning move a batch to CUDA?**

A. Yes  
B. No  
C. Only labels  
D. Only models


**Q8. Correct tensor `.to` pattern?**

A. `x.to(device)` only  
B. `x = x.to(device)`  
C. `device.to(x)`  
D. `torch.device(x)`


**Q9. Safe gradient tensor to NumPy?**

A. `x.numpy()`  
B. `x.cuda().numpy()`  
C. `x.detach().cpu().numpy()`  
D. `numpy(x)`


**Q10. First easy OOM response?**

A. Larger batch  
B. Smaller batch  
C. More printing  
D. More histories


**Q11. `empty_cache()` frees live x?**

A. Always  
B. Never while referenced  
C. Only FP16  
D. Only CPU


**Q12. `memory_allocated()` measures?**

A. Live CUDA tensor memory  
B. Disk cache  
C. CPU RAM  
D. All process memory


**Q13. `memory_reserved()` measures?**

A. Parameters only  
B. PyTorch allocator reservation  
C. Gradients only  
D. Pinned memory


**Q14. Why use inference mode?**

A. Enable gradients  
B. Reduce Autograd overhead  
C. Move to CUDA  
D. Increase precision


**Q15. What changes Dropout behavior?**

A. `model.eval()`  
B. `no_grad()`  
C. `.cpu()`  
D. `empty_cache()`


**Q16. FP16 key risk?**

A. No GPU support ever  
B. Narrow range  
C. Eight-byte storage  
D. No matrix multiplication


**Q17. BF16 advantage over FP16?**

A. More mantissa bits  
B. Larger exponent range  
C. One-byte storage  
D. Always faster


**Q18. Autocast purpose?**

A. Delete graphs  
B. Choose op precision  
C. Pin batches  
D. Select GPU


**Q19. GradScaler primarily helps?**

A. FP16 gradient underflow  
B. CPU transfer  
C. Model placement  
D. Data shuffling


**Q20. Accurate Python CUDA timing needs?**

A. More `.item()`  
B. Synchronization  
C. `empty_cache()`  
D. CPU model


**Q21. Frequent `.item()` may cause?**

A. Asynchrony  
B. Synchronization  
C. OOM always  
D. Model conversion


**Q22. Larger batches generally create?**

A. Fewer activations  
B. More activation memory  
C. No change  
D. CPU-only models


**Q23. Saving `loss` tensors may retain?**

A. Dataset  
B. Computation graphs  
C. Drivers  
D. Pinned pages only


**Q24. Portable pin-memory setting?**

A. Always false  
B. `torch.cuda.is_available()`  
C. `device == 1`  
D. `model.training`


**Q25. `non_blocking=True` guarantees speedup?**

A. Yes  
B. No  
C. Only CPU  
D. Only MPS


**Q26. Model parameters move with?**

A. `model.to(device)`  
B. `X.to(device)`  
C. `loader.to(device)`  
D. `loss.to(device)`


**Q27. Longer transformer sequences tend to?**

A. Reduce activation memory  
B. Increase memory  
C. Disable attention  
D. Force CPU


**Q28. Parameter memory helper omits?**

A. Parameter tensors  
B. Activations/optimizer state  
C. Element size  
D. Parameter count


**Q29. A tiny model may be slower on GPU because?**

A. GPU cannot add  
B. Overheads dominate  
C. CUDA disables math  
D. FP32 unavailable


**Q30. Best quick mismatch inspection?**

A. Print devices  
B. Clear cache  
C. Increase batch  
D. Convert everything to FP16


### Quiz answer key

1:C · 2:C · 3:B · 4:B · 5:C · 6:A · 7:B · 8:B · 9:C · 10:B · 11:B · 12:A · 13:B · 14:B · 15:A · 16:B · 17:B · 18:B · 19:A · 20:B · 21:B · 22:B · 23:B · 24:B · 25:B · 26:A · 27:B · 28:B · 29:B · 30:A


## 35. Write From Memory

Write each snippet without looking back. The reference cell below is executable on CPU-only systems.

1. Check CUDA availability.
2. Detect Apple MPS safely.
3. Select CUDA, MPS, or CPU.
4. Move a model to `device`.
5. Move `X` and `y` to `device`.
6. Print a tensor's device.
7. Move a tensor to CPU.
8. Convert a gradient-tracking GPU tensor to NumPy safely.
9. Build a DataLoader with conditional pinned memory.
10. Use a conditional non-blocking transfer.
11. Put a model in evaluation mode.
12. Use `torch.no_grad()`.
13. Use `torch.inference_mode()`.
14. Calculate tensor storage.
15. Inspect allocated CUDA memory conditionally.
16. Clear unused CUDA cache conditionally.
17. Write a guarded CUDA autocast forward pass.
18. Write the AMP scaling sequence.
19. Print the first model parameter's device.
20. Return a detached CPU output for storage.


In [ ]:
# Safe reference implementations for the write-from-memory prompts.
cuda_available = torch.cuda.is_available()
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
memory_device = torch.device("cuda" if cuda_available else "mps" if mps_available else "cpu")
memory_model = nn.Linear(4, 2).to(memory_device)
memory_X = torch.randn(3, 4).to(memory_device)
memory_y = torch.randint(0, 2, (3,)).to(memory_device)
print(memory_X.device)
memory_X_cpu = memory_X.cpu()
memory_array = memory_model(memory_X).detach().cpu().numpy()
memory_loader = DataLoader(
    TensorDataset(torch.randn(8, 4), torch.randint(0, 2, (8,))),
    batch_size=4, pin_memory=cuda_available,
)
ref_X, ref_y = next(iter(memory_loader))
ref_X = ref_X.to(memory_device, non_blocking=cuda_available)
ref_y = ref_y.to(memory_device, non_blocking=cuda_available)
memory_model.eval()
with torch.no_grad():
    ref_no_grad = memory_model(ref_X)
with torch.inference_mode():
    ref_inference = memory_model(ref_X)
storage_bytes = ref_X.numel() * ref_X.element_size()
if cuda_available:
    print(torch.cuda.memory_allocated())
    torch.cuda.empty_cache()
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        ref_amp_output = memory_model(ref_X)
else:
    print("CUDA-only memory/autocast snippets skipped safely")
print(next(memory_model.parameters()).device)
stored_output = ref_inference.detach().cpu()
assert stored_output.device.type == "cpu" and storage_bytes > 0
print("✅ All safe references executed")


### GPU Syntax Flashcards

Try to answer before expanding.


**Check CUDA availability.**

<details><summary>Answer</summary>

`torch.cuda.is_available()`

</details>


**Check MPS availability safely.**

<details><summary>Answer</summary>

`hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()`

</details>


**Represent the CPU device.**

<details><summary>Answer</summary>

`torch.device('cpu')`

</details>


**Move a model.**

<details><summary>Answer</summary>

`model = model.to(device)`

</details>


**Move tensor X.**

<details><summary>Answer</summary>

`X = X.to(device)`

</details>


**Move a target.**

<details><summary>Answer</summary>

`y = y.to(device)`

</details>


**Inspect X placement.**

<details><summary>Answer</summary>

`X.device`

</details>


**Move X to CPU.**

<details><summary>Answer</summary>

`X = X.cpu()`

</details>


**Safe tensor-to-NumPy conversion.**

<details><summary>Answer</summary>

`X.detach().cpu().numpy()`

</details>


**Disable gradients for inference.**

<details><summary>Answer</summary>

`with torch.no_grad():`

</details>


**Use optimized inference context.**

<details><summary>Answer</summary>

`with torch.inference_mode():`

</details>


**Conditional pinned memory.**

<details><summary>Answer</summary>

`pin_memory=torch.cuda.is_available()`

</details>


**Allow a non-blocking move.**

<details><summary>Answer</summary>

`X = X.to(device, non_blocking=True)`

</details>


**Compute tensor storage bytes.**

<details><summary>Answer</summary>

`X.numel() * X.element_size()`

</details>


**Inspect live CUDA tensor memory.**

<details><summary>Answer</summary>

`torch.cuda.memory_allocated()`

</details>


## Performance Debugging Scenarios

1. **20% utilization, mostly empty GPU, gaps between batches:** check DataLoader/preprocessing/storage, batch size, and CPU bottlenecks.
2. **OOM after hundreds of iterations:** inspect histories such as `history.append(loss)` and saved GPU outputs.
3. **Tiny model slower on GPU:** launch/transfer overhead dominates.
4. **Batch 16 works, 256 OOMs:** reduce batch, use AMP, or accumulate gradients.
5. **Inference memory unexpectedly high:** verify `eval()` plus `inference_mode()`/`no_grad()` and avoid stored GPU outputs.
6. **FP16 gives no speedup:** confirm hardware kernels/support and find the real pipeline bottleneck.
7. **Timing varies on first run:** warm up before repeated measurement.
8. **Pinned memory has no effect:** verify CUDA copies, non-blocking use, and overlap; benchmark end to end.
9. **GPU stalls whenever metrics log:** frequent `.item()`/printing may synchronize.
10. **Reserved memory is much larger than allocated:** the caching allocator holds reusable blocks; this alone is not a leak.


## 36. Final GPU-Aware Training Challenge 🔴 Hard

Build and train a complete 5-class classifier. The completed solution below is intentionally small, portable, and assert-validated.


In [ ]:
torch.manual_seed(42)

challenge_X = torch.randn(2000, 64)
true_weights = torch.randn(64, 5)
scores = challenge_X @ true_weights
challenge_y = scores.argmax(dim=1)

train_X, val_X = challenge_X[:1600], challenge_X[1600:]
train_y, val_y = challenge_y[:1600], challenge_y[1600:]

if torch.cuda.is_available():
    challenge_device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    challenge_device = torch.device("mps")
else:
    challenge_device = torch.device("cpu")
assert challenge_device.type in {"cuda", "mps", "cpu"}
print("Using device:", challenge_device)

challenge_train_loader = DataLoader(
    TensorDataset(train_X, train_y), batch_size=64, shuffle=True,
    pin_memory=torch.cuda.is_available(),
)
challenge_val_loader = DataLoader(
    TensorDataset(val_X, val_y), batch_size=128, shuffle=False,
    pin_memory=torch.cuda.is_available(),
)

challenge_model = nn.Sequential(
    nn.Linear(64, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 5),
).to(challenge_device)
challenge_loss_fn = nn.CrossEntropyLoss()
challenge_optimizer = optim.Adam(challenge_model.parameters(), lr=0.01)

assert next(challenge_model.parameters()).device.type == challenge_device.type
assert challenge_model(torch.randn(2, 64, device=challenge_device)).shape == (2, 5)
print("✅ Data, loaders, device, and model ready")


In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = total_correct = total_examples = 0
    use_non_blocking = device.type == "cuda"
    for X, y in loader:
        X = X.to(device, non_blocking=use_non_blocking)
        y = y.to(device, non_blocking=use_non_blocking)
        optimizer.zero_grad()
        logits = model(X)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total_examples += X.size(0)
    return total_loss / total_examples, total_correct / total_examples


def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = total_correct = total_examples = 0
    use_non_blocking = device.type == "cuda"
    with torch.inference_mode():
        for X, y in loader:
            X = X.to(device, non_blocking=use_non_blocking)
            y = y.to(device, non_blocking=use_non_blocking)
            logits = model(X)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * X.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total_examples += X.size(0)
    return total_loss / total_examples, total_correct / total_examples


for epoch in range(3):
    train_loss, train_accuracy = train_one_epoch(
        challenge_model, challenge_train_loader, challenge_loss_fn,
        challenge_optimizer, challenge_device,
    )
    val_loss, val_accuracy = evaluate(
        challenge_model, challenge_val_loader, challenge_loss_fn, challenge_device,
    )
    print(
        f"Epoch {epoch + 1}: train loss={train_loss:.4f}, "
        f"train acc={train_accuracy:.3f}, val loss={val_loss:.4f}, "
        f"val acc={val_accuracy:.3f}"
    )

assert 0 <= train_accuracy <= 1 and 0 <= val_accuracy <= 1
assert train_loss >= 0 and val_loss >= 0
print("✅ Final training challenge completed")


### Optional CUDA mixed-precision version

Replace the ordinary forward/backward in `train_one_epoch` with autocast and `torch.amp.GradScaler("cuda")` only when `device.type == "cuda"`. CPU/MPS completion does not require AMP.


In [ ]:
if challenge_device.type == "cuda":
    print("Allocated MB:", bytes_to_mb(torch.cuda.memory_allocated()))
    print("Reserved MB:", bytes_to_mb(torch.cuda.memory_reserved()))
    print("Peak allocated MB:", bytes_to_mb(torch.cuda.max_memory_allocated()))
else:
    print("CUDA memory metrics unavailable on this device.")


### Final tensor-memory calculation

Calculate the storage of a `(64, 3, 224, 224)` FP32 image batch. Predict the FP16 result first.


In [ ]:
image_batch = torch.randn(64, 3, 224, 224)
image_bytes = image_batch.numel() * image_batch.element_size()
image_half_bytes = image_batch.to(torch.float16).numel() * image_batch.to(torch.float16).element_size()
print(f"FP32: {bytes_to_mb(image_bytes):.2f} MiB")
print(f"FP16: {bytes_to_mb(image_half_bytes):.2f} MiB")
assert image_half_bytes * 2 == image_bytes
print("✅ FP16 tensor storage is approximately halved")


### 🔴 Optional hard challenge: find a safe batch size

In production, benchmark a **predefined, conservative** candidate list, catch `torch.cuda.OutOfMemoryError`, clear references, and stop at the last successful candidate. Do not allocate arbitrary exponentially growing tensors or intentionally crash a shared notebook/kernel. Measure throughput as well as memory; the largest fitting batch is not automatically fastest.


## 37. Cheat Sheet

```python
# Device selection
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = model.to(device)
X = X.to(device)
y = y.to(device)
print(X.device)
X = X.cpu()
array = X.detach().cpu().numpy()

loader = DataLoader(
    dataset, batch_size=64, shuffle=True,
    pin_memory=torch.cuda.is_available(),
)

for X, y in loader:
    X = X.to(device, non_blocking=torch.cuda.is_available())
    y = y.to(device, non_blocking=torch.cuda.is_available())
    optimizer.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()

model.eval()
with torch.inference_mode():
    output = model(X)

if torch.cuda.is_available():
    print(torch.cuda.memory_allocated())
    print(torch.cuda.memory_reserved())
    print(torch.cuda.max_memory_allocated())

memory_bytes = x.numel() * x.element_size()

with torch.autocast(device_type="cuda", dtype=torch.float16):
    output = model(X)

scaler = torch.amp.GradScaler("cuda")  # modern API
```


### Critical mental models

```text
MODEL DEVICE = INPUT DEVICE = TARGET DEVICE

Dataset / DataLoader → CPU batch → .to(device) → GPU/MPS model → compute

Training memory = parameters + activations + gradients + optimizer state
                + batch + temporary buffers

OOM? → reduce batch → mixed precision → gradient accumulation
     → remove graph retention → advanced techniques if needed
```


### Interview quick reference

| API/concept | Meaning |
|---|---|
| `torch.cuda.is_available()` | check CUDA usability |
| `torch.backends.mps.is_available()` | check Apple MPS |
| `.to(device)` | move tensor/model |
| `.cpu()` | move tensor to CPU |
| `pin_memory` | potentially faster CPU→CUDA source memory |
| `non_blocking` | permit asynchronous transfer when possible |
| `torch.no_grad()` | disable gradient calculation |
| `torch.inference_mode()` | inference-optimized no-grad mode |
| FP16 | 16-bit, narrow range |
| BF16 | 16-bit, larger exponent range |
| autocast | choose operation precision automatically |
| GradScaler | help FP16 gradients avoid underflow |
| `memory_allocated` | live CUDA tensor memory |
| `memory_reserved` | CUDA memory reserved by PyTorch |
| `empty_cache` | release unused cached blocks, not live tensors |


## Before Moving to 09_debugging_and_performance.ipynb

- [ ] I can check whether CUDA is available.
- [ ] I can detect Apple MPS.
- [ ] I can select a portable device.
- [ ] I can move a tensor and model to a device.
- [ ] I understand device mismatch errors and can inspect devices.
- [ ] I can move outputs to CPU and safely convert them to NumPy.
- [ ] I understand CPU↔GPU transfer cost.
- [ ] I understand `pin_memory` and `non_blocking=True`.
- [ ] I know what consumes GPU memory and why training uses more than inference.
- [ ] I understand CUDA OOM causes and first responses.
- [ ] I understand `empty_cache()` and graph-retention leaks.
- [ ] I can calculate tensor memory with `numel()` and `element_size()`.
- [ ] I understand FP32, FP16, BF16, mixed precision, autocast, and scaling.
- [ ] I know CUDA timing may require synchronization.
- [ ] I can write a device-agnostic training loop.

You should now be able to diagnose device mismatch, unnecessary transfers, GPU-memory pressure, incorrect inference mode, low-precision issues, and data-transfer bottlenecks.
